## 1. Instalación de Dependencias

Solo si no las tienes instaladas:

In [29]:
# Descomentar si necesitas instalar:
# !pip install openai faiss-cpu python-dotenv

## 2. Importar Librerías

Las mismas que usas en `retrieval_system.py`

In [30]:
import json
import os
import time
import numpy as np
import faiss
import pickle
from typing import List, Dict, Any, Optional
from openai import OpenAI
from dotenv import load_dotenv

# Cargar variables de entorno (.env)
load_dotenv()

print("Librerias importadas correctamente")

Librerias importadas correctamente


## 3. Configurar OpenAI

Mismo modelo que usas para CDMX: `text-embedding-3-small`

In [31]:
# Obtener API key desde .env
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if not OPENAI_API_KEY:
    print("Error: OPENAI_API_KEY no encontrada en .env")
    print("Crea un archivo .env con: OPENAI_API_KEY=tu-api-key")
else:
    print("OpenAI API Key cargada")
    client = OpenAI(api_key=OPENAI_API_KEY)
    
# Configuración
EMBEDDING_MODEL = 'text-embedding-3-small'  # Mismo que CDMX
EMBEDDING_DIMENSION = 1536  # Dimensión del modelo

print(f"Modelo de embeddings: {EMBEDDING_MODEL}")
print(f"Dimension de vectores: {EMBEDDING_DIMENSION}")

OpenAI API Key cargada
Modelo de embeddings: text-embedding-3-small
Dimension de vectores: 1536


## 4. Cargar Datos de Especialistas

In [32]:
# Cargar JSON
DATA_PATH = 'data/mental_health_specialists_cleaned.json'

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    specialists = json.load(f)

print(f"Cargados {len(specialists):,} especialistas")
print(f"\nEjemplo del primer especialista:")
print(f"   NPI: {specialists[0]['npi']}")
print(f"   Nombre: {specialists[0]['name']}")
print(f"   Especialidad: {specialists[0]['primary_taxonomy']}")

Cargados 4,291 especialistas

Ejemplo del primer especialista:
   NPI: 1417367343
   Nombre: GARY T GOSSINGER MD PA
   Especialidad: Psychiatry & Neurology, Psychiatry


## 5. Función para Crear Descripciones

Similar a `_create_specialist_text()` en `retrieval_system.py`

In [33]:
def create_specialist_description(specialist: Dict[str, Any]) -> str:
    """
    Crea una descripción rica del especialista para embeddings.
    Combina toda la información relevante para búsqueda semántica.
    
    Igual al patrón de retrieval_system.py
    """
    parts = []
    
    # Nombre y credencial
    if specialist.get('name'):
        parts.append(specialist['name'])
    if specialist.get('credential'):
        parts.append(specialist['credential'])
    
    # Tipo de proveedor (Individual u Organization)
    if specialist.get('provider_type'):
        parts.append(specialist['provider_type'])
    
    # Especialidad principal (MUY IMPORTANTE)
    if specialist.get('primary_taxonomy'):
        parts.append(specialist['primary_taxonomy'])
    
    # Todas las taxonomías (puede tener múltiples especialidades)
    if specialist.get('all_taxonomies'):
        try:
            taxonomies = json.loads(specialist['all_taxonomies'])
            for tax in taxonomies:
                if tax.get('desc'):
                    parts.append(tax['desc'])
        except:
            pass
    
    # Ubicación (ciudad y estado para búsquedas geográficas)
    if specialist.get('practice_location'):
        try:
            location = json.loads(specialist['practice_location'])
            city = location.get('city', '')
            state = location.get('state', '')
            if city:
                parts.append(city)
            if state:
                parts.append(state)
        except:
            pass
    
    # Unir todo con espacios
    return ' '.join([p for p in parts if p])

# Prueba con 3 especialistas
print("EJEMPLOS DE DESCRIPCIONES:")
print("=" * 70)
for i in range(3):
    desc = create_specialist_description(specialists[i])
    print(f"\n{i+1}. {specialists[i]['npi']}")
    print(f"   {desc[:150]}...")
    print(f"   Longitud: {len(desc)} caracteres")

EJEMPLOS DE DESCRIPCIONES:

1. 1417367343
   GARY T GOSSINGER MD PA Organization Psychiatry & Neurology, Psychiatry Psychiatry & Neurology, Psychiatry GAINESVILLE FL...
   Longitud: 120 caracteres

2. 1134520182
   1 CP PLACE PLLC Organization Psychiatry & Neurology, Neurology with Special Qualifications in Child Neurology Psychiatry & Neurology, Neurology with S...
   Longitud: 199 caracteres

3. 1972171536
   1 LIFE NEUROLOGY CENTER Organization Psychiatry & Neurology, Neurology Psychiatry & Neurology, Neurology MUSCLE SHOALS AL...
   Longitud: 121 caracteres


## 6. Generar Embeddings con OpenAI

**¡IMPORTANTE!** Esto puede tomar unos minutos para 60K+ especialistas.

Igual que en `retrieval_system.py`: procesamos en batches de 50.

In [34]:
def generate_embeddings(specialists: List[Dict], batch_size: int = 50) -> np.ndarray:
    """
    Genera embeddings usando OpenAI API en batches.
    
    Args:
        specialists: Lista de especialistas
        batch_size: Tamaño del batch (50 es seguro para OpenAI)
        
    Returns:
        Array numpy con embeddings (shape: [n_specialists, 1536])
    """
    # Crear descripciones
    print("Creando descripciones...")
    texts = [create_specialist_description(s) for s in specialists]
    
    # Generar embeddings en batches
    all_embeddings = []
    total_batches = (len(texts) + batch_size - 1) // batch_size
    
    print(f"\nGenerando embeddings en {total_batches} batches de {batch_size}...")
    print("Esto puede tomar varios minutos...\n")
    
    start_time = time.time()
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        batch_num = i // batch_size + 1
        
        # Reintentos en caso de error
        for attempt in range(3):
            try:
                response = client.embeddings.create(
                    model=EMBEDDING_MODEL,
                    input=batch
                )
                batch_embeddings = [d.embedding for d in response.data]
                all_embeddings.extend(batch_embeddings)
                
                # Mostrar progreso
                elapsed = time.time() - start_time
                progress = (batch_num / total_batches) * 100
                print(f"Batch {batch_num}/{total_batches} ({progress:.1f}%) - {elapsed:.1f}s")
                
                # Pequeña pausa para no sobrecargar la API
                time.sleep(0.1)
                break
                
            except Exception as e:
                if attempt == 2:
                    print(f"Error en batch {batch_num}: {e}")
                    raise
                print(f"Reintento {attempt + 1}/3")
                time.sleep(1 + attempt)
    
    total_time = time.time() - start_time
    print(f"\nEmbeddings generados en {total_time:.1f} segundos")
    print(f"Promedio: {total_time/total_batches:.2f}s por batch")
    
    return np.array(all_embeddings, dtype='float32')

# Info antes de generar
print(f"Total de especialistas: {len(specialists):,}")
print(f"Batches estimados: {(len(specialists) + 49) // 50}")
print(f"Tiempo estimado: ~{((len(specialists) + 49) // 50) * 0.5:.0f} segundos\n")
print("NOTA: Si ya generaste embeddings antes, puedes saltarte este paso y cargar el cache.")

Total de especialistas: 4,291
Batches estimados: 86
Tiempo estimado: ~43 segundos

NOTA: Si ya generaste embeddings antes, puedes saltarte este paso y cargar el cache.


## 7. Crear Índice FAISS

FAISS = Facebook AI Similarity Search

Igual que en `retrieval_system.py`: usamos `IndexFlatIP` (Inner Product) para cosine similarity.

In [35]:
def create_faiss_index(embeddings: np.ndarray) -> faiss.IndexFlatIP:
    """
    Crea índice FAISS para búsqueda rápida.
    Usa Inner Product (IP) para cosine similarity.
    
    Args:
        embeddings: Embeddings normalizados
        
    Returns:
        Índice FAISS listo para búsqueda
    """
    dimension = embeddings.shape[1]
    print(f"Dimension de embeddings: {dimension}")
    
    # Normalizar embeddings para cosine similarity
    print("Normalizando embeddings...")
    faiss.normalize_L2(embeddings)
    
    # Crear índice
    print("Creando indice FAISS...")
    index = faiss.IndexFlatIP(dimension)  # Inner Product
    
    # Agregar vectores
    print("Agregando vectores al indice...")
    index.add(embeddings)
    
    print(f"\nIndice creado exitosamente")
    print(f"   Total de vectores: {index.ntotal:,}")
    print(f"   Dimension: {dimension}")
    print(f"   Tipo: IndexFlatIP (cosine similarity)")
    
    return index

# Crear índice (después de generar embeddings)
# index = create_faiss_index(embeddings)

## 8. Guardar Cache (para no regenerar)

Igual que en `retrieval_system.py`: guardamos embeddings y metadata en disco.

In [36]:
def save_cache(index, specialists, cache_dir='faiss_nppes'):
    """
    Guarda índice FAISS y metadatos en disco.
    
    Args:
        index: Índice FAISS
        specialists: Lista de especialistas
        cache_dir: Directorio donde guardar
    """
    # Crear directorio
    os.makedirs(cache_dir, exist_ok=True)
    
    # Guardar índice FAISS
    index_path = os.path.join(cache_dir, 'nppes_index.bin')
    print(f"Guardando indice FAISS en {index_path}...")
    faiss.write_index(index, index_path)
    
    # Guardar metadata (especialistas)
    metadata_path = os.path.join(cache_dir, 'nppes_metadata.pkl')
    print(f"Guardando metadata en {metadata_path}...")
    with open(metadata_path, 'wb') as f:
        pickle.dump({'specialists': specialists}, f)
    
    print(f"\nCache guardado exitosamente en '{cache_dir}/'")
    print(f"   {index_path}")
    print(f"   {metadata_path}")

# Guardar (después de crear índice)
# save_cache(index, specialists)

## 9. Cargar Cache (rápido)

Si ya generaste embeddings antes, carga el cache en segundos.

In [37]:
def load_cache(cache_dir='faiss_nppes'):
    """
    Carga índice FAISS y metadatos desde disco.
    
    Returns:
        (index, specialists): Índice FAISS y lista de especialistas
    """
    index_path = os.path.join(cache_dir, 'nppes_index.bin')
    metadata_path = os.path.join(cache_dir, 'nppes_metadata.pkl')
    
    if not os.path.exists(index_path) or not os.path.exists(metadata_path):
        print(f"Cache no encontrado en '{cache_dir}/'")
        print(f"Primero genera embeddings y guarda el cache.")
        return None, None
    
    print(f"Cargando cache desde '{cache_dir}/'...")
    
    # Cargar índice
    print(f"   Cargando indice FAISS...")
    index = faiss.read_index(index_path)
    
    # Cargar metadata
    print(f"   Cargando metadata...")
    with open(metadata_path, 'rb') as f:
        cached_data = pickle.load(f)
        specialists = cached_data['specialists']
    
    print(f"\nCache cargado exitosamente")
    print(f"   Especialistas: {len(specialists):,}")
    print(f"   Vectores en indice: {index.ntotal:,}")
    
    return index, specialists

# Cargar cache
index, specialists = load_cache()

if index is None:
    print("\nEjecuta las celdas anteriores para generar embeddings.")

Cache no encontrado en 'faiss_nppes/'
Primero genera embeddings y guarda el cache.

Ejecuta las celdas anteriores para generar embeddings.


## 9.1 Cargar o Generar Embeddings

Esta celda verifica si existe cache. Si no existe, genera los embeddings desde cero (esto toma varios minutos y consume API credits de OpenAI).

In [38]:
# Intentar cargar cache existente
index, specialists = load_cache()

# Si no existe cache, generar embeddings
if index is None or specialists is None:
    print("\n" + "="*60)
    print("GENERANDO EMBEDDINGS (primera vez)")
    print("="*60)
    print("\nEsto puede tardar varios minutos y consume creditos de OpenAI.")
    print("Los embeddings se guardaran en cache para futuras sesiones.\n")
    
    # Cargar datos
    with open(DATA_PATH, 'r', encoding='utf-8') as f:
        specialists = json.load(f)
    print(f"Especialistas cargados: {len(specialists):,}")
    
    # Generar embeddings
    embeddings = generate_embeddings(specialists)
    
    # Crear indice FAISS
    index = create_faiss_index(embeddings)
    
    # Guardar cache para futuras sesiones
    save_cache(index, specialists)
    
    print("\n" + "="*60)
    print("EMBEDDINGS GENERADOS Y GUARDADOS")
    print("="*60)
else:
    print("\n" + "="*60)
    print("CACHE CARGADO EXITOSAMENTE")
    print("="*60)

# Verificar estado
print(f"\nEstado del sistema:")
print(f"   Especialistas: {len(specialists) if specialists else 0:,}")
print(f"   Vectores FAISS: {index.ntotal if index else 0:,}")

Cache no encontrado en 'faiss_nppes/'
Primero genera embeddings y guarda el cache.

GENERANDO EMBEDDINGS (primera vez)

Esto puede tardar varios minutos y consume creditos de OpenAI.
Los embeddings se guardaran en cache para futuras sesiones.

Especialistas cargados: 4,291
Creando descripciones...

Generando embeddings en 86 batches de 50...
Esto puede tomar varios minutos...

Batch 1/86 (1.2%) - 4.2s
Batch 2/86 (2.3%) - 5.4s
Batch 3/86 (3.5%) - 6.1s
Batch 4/86 (4.7%) - 8.0s
Batch 5/86 (5.8%) - 9.5s
Batch 6/86 (7.0%) - 10.3s
Batch 7/86 (8.1%) - 11.1s
Batch 8/86 (9.3%) - 12.3s
Batch 9/86 (10.5%) - 14.1s
Batch 10/86 (11.6%) - 15.6s
Batch 11/86 (12.8%) - 16.3s
Batch 12/86 (14.0%) - 17.3s
Batch 13/86 (15.1%) - 18.1s
Batch 14/86 (16.3%) - 19.3s
Batch 15/86 (17.4%) - 20.2s
Batch 16/86 (18.6%) - 21.8s
Batch 17/86 (19.8%) - 22.7s
Batch 18/86 (20.9%) - 23.4s
Batch 19/86 (22.1%) - 24.3s
Batch 20/86 (23.3%) - 25.0s
Batch 21/86 (24.4%) - 25.7s
Batch 22/86 (25.6%) - 27.6s
Batch 23/86 (26.7%) - 28.3

## 10. Función de Búsqueda

Igual que en `retrieval_system.py`: buscar especialistas por query.

In [46]:
def search_specialists(query: str, 
                      index, 
                      specialists: List[Dict],
                      top_k: int = 5) -> List[Dict]:
    """
    Busca especialistas relevantes para la query del usuario.
    
    Args:
        query: Pregunta o descripción del usuario
        index: Índice FAISS
        specialists: Lista de especialistas
        top_k: Número de resultados a devolver
        
    Returns:
        Lista de especialistas ordenados por relevancia
    """
    # 1. Generar embedding de la query
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=[query]
    )
    query_embedding = np.array(response.data[0].embedding, dtype='float32').reshape(1, -1)
    
    # 2. Normalizar para cosine similarity
    faiss.normalize_L2(query_embedding)
    
    # 3. Buscar en FAISS
    similarities, indices = index.search(query_embedding, top_k)
    
    # 4. Construir resultados
    results = []
    for idx, similarity in zip(indices[0], similarities[0]):
        specialist = specialists[idx].copy()
        specialist['similarity_score'] = float(similarity)
        
        # Extraer ubicación
        try:
            location = json.loads(specialist.get('practice_location', '{}'))
            specialist['city'] = location.get('city', 'N/A')
            specialist['state'] = location.get('state', 'N/A')
            specialist['telephone'] = location.get('telephone', 'N/A')
        except:
            specialist['city'] = 'N/A'
            specialist['state'] = 'N/A'
            specialist['telephone'] = 'N/A'
        
        results.append(specialist)
    
    return results


## 11. Formatear Resultados

Mostrar resultados de forma clara y legible.

In [47]:
def format_results(results: List[Dict]):
    """
    Formatea resultados para mostrar al usuario.
    
    Args:
        results: Lista de especialistas del search
    """
    print("\n" + "="*70)
    print(f"RESULTADOS DE BUSQUEDA ({len(results)} especialistas)")
    print("="*70)
    
    for i, specialist in enumerate(results, 1):
        print(f"\n{i}. {specialist['name']}")
        print(f"   {'-'*66}")
        print(f"   NPI: {specialist['npi']}")
        print(f"   Tipo: {specialist['provider_type']}")
        print(f"   Especialidad: {specialist['primary_taxonomy']}")
        print(f"   Ubicacion: {specialist['city']}, {specialist['state']}")
        print(f"   Telefono: {specialist['telephone']}")
        print(f"   Similitud: {specialist['similarity_score']:.4f}")
    
    print("\n" + "="*70)


## 12. Pruebas del Sistema

Probemos el sistema con diferentes queries.

In [41]:
# Verificar que tenemos todo listo
if index is None or specialists is None:
    print("Error: Primero carga el cache o genera embeddings")
else:
    print("Sistema listo para busquedas")
    print(f"   Especialistas indexados: {len(specialists):,}")
    print(f"   Vectores en FAISS: {index.ntotal:,}")

Sistema listo para busquedas
   Especialistas indexados: 4,291
   Vectores en FAISS: 4,291


### Prueba 1: Buscar Psiquiatra para Ansiedad en California

In [42]:
query1 = "psychiatrist for anxiety and depression in California"

print(f"Query: '{query1}'")
results1 = search_specialists(query1, index, specialists, top_k=5)
format_results(results1)

Query: 'psychiatrist for anxiety and depression in California'

RESULTADOS DE BUSQUEDA (5 especialistas)

1. ABC ANXIETY
   ------------------------------------------------------------------
   NPI: 1407354897
   Tipo: Organization
   Especialidad: Psychiatry & Neurology, Psychiatry
   Ubicacion: LOS ANGELES, CA
   Telefono: 310-943-6983
   Similitud: 0.5977

2. CRISELDA ABAD-SANTOS
   ------------------------------------------------------------------
   NPI: 1760461826
   Tipo: Individual
   Especialidad: Psychiatry & Neurology, Psychiatry
   Ubicacion: WOODLAND HILLS, CA
   Telefono: 818-992-3121
   Similitud: 0.5726

3. ABS PSYCHOLOGY SERVICES CALIFORNIA
   ------------------------------------------------------------------
   NPI: 1336659002
   Tipo: Organization
   Especialidad: Community/Behavioral Health
   Ubicacion: RANCHO CUCAMONGA, CA
   Telefono: 858-264-5858
   Similitud: 0.5677

4. ACCESS PSYCHIATRIC CARE
   -----------------------------------------------------------------

### Prueba 2: Psicólogo Infantil en New York

In [43]:
query2 = "child psychologist in New York for ADHD"

print(f"Query: '{query2}'")
results2 = search_specialists(query2, index, specialists, top_k=5)
format_results(results2)

Query: 'child psychologist in New York for ADHD'

RESULTADOS DE BUSQUEDA (5 especialistas)

1. ADHD NEW YORK, LLC
   ------------------------------------------------------------------
   NPI: 1326312059
   Tipo: Organization
   Especialidad: Psychiatry & Neurology, Psychiatry
   Ubicacion: NEW YORK, NY
   Telefono: 212-799-7777
   Similitud: 0.7689

2. ADRIA ADAMS
   ------------------------------------------------------------------
   NPI: 1619022860
   Tipo: Individual
   Especialidad: Psychologist
   Ubicacion: NEW YORK, NY
   Telefono: 646-430-2827
   Similitud: 0.6282

3. DANIEL ADLER
   ------------------------------------------------------------------
   NPI: 1548384209
   Tipo: Individual
   Especialidad: Psychiatry & Neurology, Neurology with Special Qualifications in Child Neurology
   Ubicacion: NEW YORK, NY
   Telefono: 201-894-1551
   Similitud: 0.6135

4. ACUTE BEHAVIORAL HEALTH OF NEW JERSEY, P.A.
   ------------------------------------------------------------------
   N

### Prueba 3: Neurólogo para Demencia en Texas

In [44]:
query3 = "neurologist specializing in dementia in Texas"

print(f"Query: '{query3}'")
results3 = search_specialists(query3, index, specialists, top_k=5)
format_results(results3)

Query: 'neurologist specializing in dementia in Texas'

RESULTADOS DE BUSQUEDA (5 especialistas)

1. MARK ALBERTS
   ------------------------------------------------------------------
   NPI: 1992795140
   Tipo: Individual
   Especialidad: Psychiatry & Neurology, Neurology
   Ubicacion: DALLAS, TX
   Telefono: 214-645-0624
   Similitud: 0.5806

2. ADVANCED TEXAS NEUROLOGY LLC
   ------------------------------------------------------------------
   NPI: 1811705585
   Tipo: Organization
   Especialidad: Psychiatry & Neurology, Neurology
   Ubicacion: MURPHY, TX
   Telefono: None
   Similitud: 0.5703

3. WAMDA AHMED
   ------------------------------------------------------------------
   NPI: 1417276635
   Tipo: Individual
   Especialidad: Psychiatry & Neurology, Neurology
   Ubicacion: HOUSTON, TX
   Telefono: 713-254-2421
   Similitud: 0.5556

4. AMIR AKHTER
   ------------------------------------------------------------------
   NPI: 1285683466
   Tipo: Individual
   Especialidad: Psyc